# Raw Signal Characterization

Baseline RF statistics from raw ultrasound acquisitions.

**Analyses:**
1. RF amplitude statistics (RMS, min, max, dynamic range)
2. SNR estimation: signal (central part) vs noise (bottom region clip)
3. Channel cross-correlation
4. Inter-participant variability
5. Session stability

## 1. Setup & Data Loading Infrastructure

In [ ]:
import os
import sys
import re
import glob
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project root
PROJECT_ROOT = Path(os.path.abspath('')).parents[1]  # analysis/dataset -> project root
sys.path.insert(0, str(PROJECT_ROOT))

# ---------- Load config ----------
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

RAW_DATA_PATH = Path(config['global_setting']['paths']['base_data_path']) / 'raw'

# Clip boundaries from config (defines signal vs noise regions)
clip_cfg = config['preprocess']['signal']['clip']
TOTAL_DEPTH_SAMPLES = clip_cfg['initial_size']           # 1996
CLIP_START = clip_cfg['samples2remove_start']             # 107
CLIP_END_REMOVE = clip_cfg['samples2remove_end']          # 589
SIGNAL_START = CLIP_START                                  # 107
SIGNAL_END = TOTAL_DEPTH_SAMPLES - CLIP_END_REMOVE         # 1407
NOISE_START = SIGNAL_END                                   # 1407 (right side = post-tissue)
NOISE_END = TOTAL_DEPTH_SAMPLES                            # 1996

print(f"Raw data path:  {RAW_DATA_PATH}")
print(f"Depth samples:  {TOTAL_DEPTH_SAMPLES}")
print(f"Signal region:  [{SIGNAL_START}:{SIGNAL_END}] ({SIGNAL_END - SIGNAL_START} samples)")
print(f"Noise region:   [{NOISE_START}:{NOISE_END}] ({NOISE_END - NOISE_START} samples)")

In [ ]:
def build_experiment_index(raw_path):
    """Walk through raw data folder, return sorted DataFrame "exp_index" of (participant, session, experiment, path)."""
    rows = []
    for p_dir in sorted(raw_path.glob('P*')):
        if not p_dir.is_dir():
            continue
        p_id = int(re.search(r'P(\d+)', p_dir.name).group(1))
        for s_dir in sorted(p_dir.glob('session*')):
            if not s_dir.is_dir():
                continue
            s_id = int(re.search(r'session(\d+)', s_dir.name).group(1))
            for e_dir in sorted(s_dir.glob('exp*')):
                if not e_dir.is_dir():
                    continue
                e_id = int(re.search(r'exp(\d+)', e_dir.name).group(1))
                rows.append({'participant': p_id, 'session': s_id,
                             'experiment': e_id, 'path': str(e_dir)})
    return pd.DataFrame(rows)

exp_index = build_experiment_index(RAW_DATA_PATH)
print(f"Experiments found: {len(exp_index)}")
print(f"Participants: {sorted(exp_index['participant'].unique())}")
print(f"Sessions/participant: {exp_index.groupby('participant')['session'].nunique().to_dict()}")
print(f"Experiments/session: {exp_index.groupby(['participant','session'])['experiment'].count().unique()}")

In [ ]:
def load_us_channels(exp_path):
    """Load all US channels for one experiment. Returns [channels, pulses, depth]"""
    ch_files = sorted(glob.glob(os.path.join(exp_path, '_US_ch*.npy')))
    if not ch_files:
        raise FileNotFoundError(f"No US files in {exp_path}")
    channels = [np.load(f) for f in ch_files]
    return np.stack(channels, axis=0)  # [channel, pulses, depth]

# Sanity first experiment
test_us = load_us_channels(exp_index.iloc[0]['path'])
print(f"Shape: {test_us.shape}, dtype: {test_us.dtype}")
print(f"Channels: {test_us.shape[0]}, Pulses: {test_us.shape[1]}, Depth: {test_us.shape[2]}")
del test_us

## 2. RF Amplitude Statistics

Per-experiment, per-channel: **RMS**, min, max, dynamic range (dB).

In [ ]:
from tqdm.notebook import tqdm

def compute_rf_stats(us_data):
    """
    Compute RF amplitude stats per channel.
    
    Args:
        us_data: [channel, pulses, depth]
    Returns:
        list of dicts, one per channel
    """
    n_channels = us_data.shape[0]
    results = []
    for ch in range(n_channels):
        signal_region = us_data[ch, :, SIGNAL_START:SIGNAL_END].astype(np.float64)
        noise_region = us_data[ch, :, NOISE_START:NOISE_END].astype(np.float64)
        
        rms_signal = np.sqrt(np.mean(signal_region ** 2))
        rms_noise = np.sqrt(np.mean(noise_region ** 2))
        abs_max = np.max(np.abs(signal_region))
        
        # Dynamic range: peak-to-noise-floor in dB
        dynamic_range_dB = 20 * np.log10(abs_max / rms_noise) if rms_noise > 0 else np.nan
        
        results.append({
            'channel': ch,
            'rms': rms_signal,
            'min': int(signal_region.min()),
            'max': int(signal_region.max()),
            'abs_max': int(abs_max),
            'dynamic_range_dB': dynamic_range_dB,
            'rms_noise': rms_noise,
        })
    return results


# ---------- Compute across all the experiments ----------
rf_stats_rows = []

for _, row in tqdm(exp_index.iterrows(), total=len(exp_index), desc="RF stats"):
    us = load_us_channels(row['path'])
    stats = compute_rf_stats(us)
    for s in stats:
        s.update({'participant': row['participant'], 'session': row['session'],
                  'experiment': row['experiment']})
        rf_stats_rows.append(s)
    del us

df_rf = pd.DataFrame(rf_stats_rows)
print(f"\nCollected {len(df_rf)} rows ({len(exp_index)} experiments × {df_rf['channel'].nunique()} channels)")
df_rf.head(6)

In [ ]:
# ---------- Summary table ----------
summary = df_rf.groupby('channel').agg(
    rms_mean=('rms', 'mean'), rms_std=('rms', 'std'),
    abs_max_mean=('abs_max', 'mean'), abs_max_std=('abs_max', 'std'),
    dyn_range_mean=('dynamic_range_dB', 'mean'), dyn_range_std=('dynamic_range_dB', 'std'),
).round(2)

print("RF Amplitude Summary (signal region, per channel):")
print(summary)

# ---------- Boxplot ----------
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric, label in zip(axes,
    ['rms', 'abs_max', 'dynamic_range_dB'],
    ['RMS amplitude', 'Peak |amplitude|', 'Dynamic range (dB)']):
    sns.boxplot(data=df_rf, x='channel', y=metric, ax=ax)
    ax.set_xlabel('Channel')
    ax.set_ylabel(label)
    ax.set_title(label)

fig.suptitle('RF Amplitude Statistics per Channel (120 experiments)', y=1.02)
plt.tight_layout()
plt.show()

## 3. SNR Estimation

Signal region `[107:1407]` vs noise region `[1407:1996]` (post-tissue, right-side clip).  
`SNR_dB = 10 · log10( rms_signal² / rms_noise² ) = 20 · log10( rms_signal / rms_noise )`


In [ ]:
# SNR from already-computed RMS values
df_rf['snr_dB'] = 20 * np.log10(df_rf['rms'] / df_rf['rms_noise'])

# ---------- Summary ----------
snr_summary = df_rf.groupby('channel').agg(
    snr_mean=('snr_dB', 'mean'), snr_std=('snr_dB', 'std'),
    snr_min=('snr_dB', 'min'), snr_max=('snr_dB', 'max'),
).round(2)
print("SNR Summary (dB) per channel:")
print(snr_summary)

# ---------- Plots ----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Boxplot per channel
sns.boxplot(data=df_rf, x='channel', y='snr_dB', ax=axes[0])
axes[0].set_ylabel('SNR (dB)')
axes[0].set_xlabel('Channel')
axes[0].set_title('SNR per Channel')

# Histogram across all channels
for ch in sorted(df_rf['channel'].unique()):
    axes[1].hist(df_rf[df_rf['channel'] == ch]['snr_dB'],
                 bins=15, alpha=0.5, label=f'Ch {ch}')
axes[1].set_xlabel('SNR (dB)')
axes[1].set_ylabel('Count')
axes[1].set_title('SNR Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

# ---------- Flag low-SNR experiments ----------
snr_threshold = snr_summary['snr_mean'].min() - 2 * snr_summary['snr_std'].max()  # 2σ below worst channel mean
low_snr = df_rf[df_rf['snr_dB'] < snr_threshold]
if len(low_snr) > 0:
    print(f"\nLow-SNR experiments (< {snr_threshold:.1f} dB):")
    print(low_snr[['participant', 'session', 'experiment', 'channel', 'snr_dB']].to_string(index=False))
else:
    print(f"\nNo experiments below {snr_threshold:.1f} dB threshold (2σ below worst channel mean)")

## 4. Channel Cross-Correlation

Per-pulse Pearson r between channel pairs over the signal region `[107:1407]`.  
Per-pulse approach reveals whether correlation is stable across time or varies.


In [ ]:
CHANNEL_PAIRS = [(0, 1), (0, 2), (1, 2)]
PAIR_NAMES = {(0, 1): 'Ch0-Ch1', (0, 2): 'Ch0-Ch2', (1, 2): 'Ch1-Ch2'}


def compute_channel_correlation(us_data):
    """
    Per-pulse Pearson r between channel pairs over signal region.

    Args:
        us_data: [channel, pulses, depth]
    Returns:
        list of dicts with r_mean, r_std per pair
    """
    # Extract signal region, float for correlation
    sig = us_data[:, :, SIGNAL_START:SIGNAL_END].astype(np.float64)  # [channel, pulses, 1300]
    n_pulses = sig.shape[1]
    results = []

    for ch_a, ch_b in CHANNEL_PAIRS:
        a = sig[ch_a]  # [pulses, 1300]
        b = sig[ch_b]

        # per-pulse Pearson r
        a_centered = a - a.mean(axis=1, keepdims=True)
        b_centered = b - b.mean(axis=1, keepdims=True)
        num = (a_centered * b_centered).sum(axis=1)
        denom = np.sqrt((a_centered**2).sum(axis=1) * (b_centered**2).sum(axis=1))
        r_per_pulse = num / np.where(denom > 0, denom, 1.0)

        results.append({
            'pair': PAIR_NAMES[(ch_a, ch_b)],
            'r_mean': np.mean(r_per_pulse),
            'r_std': np.std(r_per_pulse),
            'r_min': np.min(r_per_pulse),
            'r_max': np.max(r_per_pulse),
        })
    return results


# ---------- Compute across all experiments ----------
corr_rows = []

for _, row in tqdm(exp_index.iterrows(), total=len(exp_index), desc="Cross-correlation"):
    us = load_us_channels(row['path'])
    corrs = compute_channel_correlation(us)
    for c in corrs:
        c.update({'participant': row['participant'], 'session': row['session'],
                  'experiment': row['experiment']})
        corr_rows.append(c)
    del us

df_corr = pd.DataFrame(corr_rows)
print(f"Collected {len(df_corr)} rows ({len(exp_index)} experiments × {len(CHANNEL_PAIRS)} pairs)")
df_corr.head(6)

In [ ]:
# ---------- Summary ----------
corr_summary = df_corr.groupby('pair').agg(
    r_mean=('r_mean', 'mean'), r_mean_std=('r_mean', 'std'),
    r_std_mean=('r_std', 'mean'),
).round(4)
print("Channel Cross-Correlation Summary:")
print(corr_summary)

# ---------- Plots ----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Boxplot: mean r per pair
sns.boxplot(data=df_corr, x='pair', y='r_mean', ax=axes[0])
axes[0].set_ylabel('Mean Pearson r (per experiment)')
axes[0].set_xlabel('Channel Pair')
axes[0].set_title('Inter-Channel Correlation')
axes[0].axhline(y=0.9, color='r', linestyle='--', alpha=0.5, label='r=0.9 (high redundancy)')
axes[0].axhline(y=0.3, color='g', linestyle='--', alpha=0.5, label='r=0.3 (low correlation)')
axes[0].legend(fontsize=8)

# Boxplot: std of r (temporal stability of correlation)
sns.boxplot(data=df_corr, x='pair', y='r_std', ax=axes[1])
axes[1].set_ylabel('Std of Pearson r (within experiment)')
axes[1].set_xlabel('Channel Pair')
axes[1].set_title('Temporal Stability of Correlation')

plt.tight_layout()
plt.show()

## 5. Inter-Participant Variability

Compare per-participant RMS amplitude distributions using:
- **KS test** (nonparametric, pairwise) — tests if distributions differ
- **Cohen's d** (pairwise) — quantifies effect size

In [ ]:
from scipy.stats import ks_2samp
from itertools import combinations

# Average RMS across channels per experiment
df_rms_avg = df_rf.groupby(['participant', 'session', 'experiment']).agg(
    rms_mean=('rms', 'mean')
).reset_index()

participants = sorted(df_rms_avg['participant'].unique())

# ---------- Pairwise tests ----------
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else 0.0

pairwise_rows = []
for p_a, p_b in combinations(participants, 2):
    rms_a = df_rms_avg[df_rms_avg['participant'] == p_a]['rms_mean'].values
    rms_b = df_rms_avg[df_rms_avg['participant'] == p_b]['rms_mean'].values

    ks_stat, ks_p = ks_2samp(rms_a, rms_b)
    d = cohens_d(rms_a, rms_b)
    effect = 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small'

    pairwise_rows.append({
        'P_a': f'P{p_a}', 'P_b': f'P{p_b}',
        'KS_stat': round(ks_stat, 4), 'KS_p': ks_p,
        'cohens_d': round(d, 3), 'effect_size': effect,
    })

df_pairwise = pd.DataFrame(pairwise_rows)
print("Pairwise Inter-Participant Comparisons (channel-averaged RMS):")
print(df_pairwise.to_string(index=False))

In [ ]:
# ---------- Violin plot ----------
fig, ax = plt.subplots(figsize=(7, 4))
sns.violinplot(data=df_rms_avg, x='participant', y='rms_mean', ax=ax, inner='box')
ax.set_xlabel('Participant')
ax.set_ylabel('RMS amplitude (channel-averaged)')
ax.set_title('Inter-Participant RMS Variability (30 experiments each)')
ax.set_xticklabels([f'P{p}' for p in participants])
plt.tight_layout()
plt.show()

# ---------- Per-participant summary ----------
part_summary = df_rms_avg.groupby('participant').agg(
    mean=('rms_mean', 'mean'), std=('rms_mean', 'std'),
    min=('rms_mean', 'min'), max=('rms_mean', 'max'),
    n=('rms_mean', 'count'),
).round(2)
part_summary.index = [f'P{p}' for p in part_summary.index]
print("\nPer-Participant RMS Summary:")
print(part_summary)

## 6. Session Stability: Hand Placement Effect & Within-Placement Repeatability

Sessions encode hand placement (not random repeats):
```
S0, S1 = table   |  S2, S3 = leg   |  S4, S5 = side
```

**6A**: --> Question: Between-placement effect — does hand position shift RF characteristics?

**6b**: --> Question: Within-placement repeatability — given same placement, how consistent are repeated sessions?

In [ ]:
# ---------- Session → Placement mapping ----------
PLACEMENT_MAP = {0: 'table', 1: 'table', 2: 'leg', 3: 'leg', 4: 'side', 5: 'side'}
PLACEMENT_PAIRS = {'table': (0, 1), 'leg': (2, 3), 'side': (4, 5)}

df_rms_avg['placement'] = df_rms_avg['session'].map(PLACEMENT_MAP)

# =============================================
# 6a: Between-Placement Effect
# =============================================
from scipy.stats import kruskal

print("=" * 60)
print("6a: HAND PLACEMENT EFFECT ON RMS")
print("=" * 60)

# Global Kruskal-Wallis across 3 placements
placement_groups = [grp['rms_mean'].values for _, grp in df_rms_avg.groupby('placement')]
h_stat, h_p = kruskal(*placement_groups)
print(f"\nGlobal Kruskal-Wallis: H={h_stat:.2f}, p={h_p:.2e}")

# Per-participant Kruskal-Wallis
print("\nPer-participant Kruskal-Wallis:")
for p in participants:
    df_p = df_rms_avg[df_rms_avg['participant'] == p]
    groups = [grp['rms_mean'].values for _, grp in df_p.groupby('placement')]
    h, pval = kruskal(*groups)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f"  P{p}: H={h:.2f}, p={pval:.3e} {sig}")

# Summary table: placement × participant
placement_summary = df_rms_avg.groupby(['participant', 'placement']).agg(
    rms_mean=('rms_mean', 'mean'), rms_std=('rms_mean', 'std'),
    n=('rms_mean', 'count'),
).round(2)
print("\nRMS by Participant × Placement:")
print(placement_summary)

In [ ]:
# ---------- 6a Plots ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Heatmap: participant × placement
pivot = df_rms_avg.groupby(['participant', 'placement'])['rms_mean'].mean().unstack()
pivot = pivot[['table', 'leg', 'side']]  # consistent order
pivot.index = [f'P{p}' for p in pivot.index]
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Mean RMS by Participant × Placement')
axes[0].set_ylabel('Participant')

# Boxplot: placement effect (all participants pooled)
sns.boxplot(data=df_rms_avg, x='placement', y='rms_mean',
            order=['table', 'leg', 'side'], ax=axes[1])
axes[1].set_xlabel('Hand Placement')
axes[1].set_ylabel('RMS amplitude (channel-averaged)')
axes[1].set_title('Hand Placement Effect on RMS')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# 6b: Within-Placement Repeatability
# =============================================
print("=" * 60)
print("6b: WITHIN-PLACEMENT REPEATABILITY")
print("=" * 60)

repeat_rows = []
for p in participants:
    for placement, (s_a, s_b) in PLACEMENT_PAIRS.items():
        rms_a = df_rms_avg[(df_rms_avg['participant'] == p) & (df_rms_avg['session'] == s_a)]['rms_mean'].values
        rms_b = df_rms_avg[(df_rms_avg['participant'] == p) & (df_rms_avg['session'] == s_b)]['rms_mean'].values

        if len(rms_a) == 0 or len(rms_b) == 0:
            continue

        # Pool both sessions for CV
        pooled = np.concatenate([rms_a, rms_b])
        cv = np.std(pooled) / np.mean(pooled) * 100  # percent

        # KS test between the two repeated sessions
        ks_stat, ks_p = ks_2samp(rms_a, rms_b)

        repeat_rows.append({
            'participant': f'P{p}', 'placement': placement,
            'session_a': s_a, 'session_b': s_b,
            'mean_a': np.mean(rms_a), 'mean_b': np.mean(rms_b),
            'CV_pct': round(cv, 2),
            'KS_stat': round(ks_stat, 4), 'KS_p': round(ks_p, 4),
        })

df_repeat = pd.DataFrame(repeat_rows)
print("\nWithin-Placement Session Repeatability:")
print(df_repeat[['participant', 'placement', 'mean_a', 'mean_b', 'CV_pct', 'KS_stat', 'KS_p']].to_string(index=False))

# Overall CV summary
print(f"\nOverall CV: mean={df_repeat['CV_pct'].mean():.1f}%, max={df_repeat['CV_pct'].max():.1f}%")
if df_repeat['CV_pct'].max() < 10:
    print("All CVs < 10% — good within-placement repeatability")
else:
    high_cv = df_repeat[df_repeat['CV_pct'] >= 10]
    print(f"High CV (>= 10%) found:")
    print(high_cv[['participant', 'placement', 'CV_pct']].to_string(index=False))

In [ ]:
# ---------- 6b Plot: Heatmap of CV ----------
pivot_cv = df_repeat.pivot(index='participant', columns='placement', values='CV_pct')
pivot_cv = pivot_cv[['table', 'leg', 'side']]

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.heatmap(pivot_cv, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax,
            vmin=0, vmax=max(15, pivot_cv.max().max()))
ax.set_title('Within-Placement CV (%) — Rule: lower = Ok more repeatable')
ax.set_ylabel('Participant')
plt.tight_layout()
plt.show()